# ScholarLens: training on free Google Colab

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom.
Run **Section 0 and 1 (everyone)**, then only the section for *your* module.
Everything is saved to your Google Drive, so a Colab disconnect will not lose trained models.

## 0. Setup (everyone)

If the install cell says **"Restart session"**, click it, then run this section again once.


In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/scholarlens.git"   # <-- change this
import os
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/scholarlens'):
    !git clone {REPO_URL} /content/scholarlens
%cd /content/scholarlens
!git pull
!pip install -q -r requirements.txt

In [ ]:
# Keep trained models and metric files on Google Drive (survives Colab disconnects).
DRIVE = "/content/drive/MyDrive/scholarlens"
for d in ("outputs", "results"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
    !rm -rf {d}
    !ln -s {DRIVE}/{d} {d}
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime type!")

## 1. Smoke test (everyone, ~2 minutes): does my data load?

In [ ]:
# Run the line for your module. Paste any error to your teammate or Claude.
!python -m src.data_loading --task summarization --max-samples 200
# !python -m src.data_loading --task qa --max-samples 200
# !python -m src.data_loading --task ner --max-samples 200

## 2. Quick end-to-end test (everyone, ~5 minutes)
Trains on 200 examples for 1 epoch to prove your whole path works before the long run.
Pick the line for your module.

In [ ]:
!python -m src.summarization.train --model t5-small --max-samples 200 --epochs 1
# !python -m src.qa.train --model distilbert-base-uncased --max-samples 200 --epochs 1
# !python -m src.ner.train --model bert-base-uncased --max-samples 200 --epochs 1

## 3A. Member 1: Summarization (BART vs T5)
Run one model per session if your GPU time is limited. Each takes roughly 1-2 hours.

In [ ]:
!python -m src.summarization.train --model facebook/bart-base

In [ ]:
!python -m src.summarization.train --model t5-small

## 3B. Member 2: Question answering (DistilBERT vs RoBERTa)

In [ ]:
!python -m src.qa.train --model distilbert-base-uncased

In [ ]:
!python -m src.qa.train --model roberta-base

## 3C. Member 3: NER (SciBERT vs BERT) and comparison tables

In [ ]:
!python -m src.ner.train --model allenai/scibert_scivocab_uncased

In [ ]:
!python -m src.ner.train --model bert-base-uncased

In [ ]:
# After all six models have results in results/, build the report tables and charts:
!python -m src.evaluation.compare
print(open('results/comparison.md').read())

## 4. Demo from Colab (Member 2)
Opens a public Gradio link you can use for the presentation. Needs trained models in `outputs/`
(or `serving.allow_fallback: true` in configs/config.yaml to preview the UI first).

In [ ]:
!python -m app.gradio_demo

## 5. Download your results
Metric files are in `results/` (also on your Drive). Upload them to the GitHub repo
(Add file > Upload files) so the whole team can build the comparison tables.

In [ ]:
!zip -qr /content/results.zip results docs 2>/dev/null
from google.colab import files
files.download('/content/results.zip')